# Fold 3 — untouched 2023 validation

Reproducible review of the single frozen Fold 3 execution. This notebook reads committed artifacts only; it does not select alerts, tune rules, or access 2024–2025 results.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
OUT = ROOT / 'outputs' / 'role_validation' / 'fold_3'
assert OUT.exists(), OUT
run = json.loads((OUT / 'run_manifest.json').read_text())
fingerprint = json.loads((OUT / 'frozen_config_fingerprint.json').read_text())
assert run['test_season'] == 2023 and not run['post_2023_results_used']
assert fingerprint['config_sha256'] == '4dcf389a1f8fcdd11a9277305a8372fadaabaa830185e07eff5d8fbb274a81c7'
run

{'stage': 'execute',
 'passed': True,
 'fold3_executed_once': True,
 'test_season': 2023,
 'post_2023_results_used': False,
 'fold4_executed': False,
 'config_sha256': '4dcf389a1f8fcdd11a9277305a8372fadaabaa830185e07eff5d8fbb274a81c7',
 'candidate_name': 'fold2_candidate_v1_symmetric_deltas',
 'partial_policies': ['PRIMARY_CONFIRMED_EXCLUDED',
  'ALL_INCLUDED',
  'STRICT_SUSPECTED_EXCLUDED'],
 'family_alert_method_rows': 1956,
 'primary_full_family_alerts': 165,
 'equal_volume_cells': 216,
 'all_equal_volume': True,
 'all_temporal_checks_passed': True,
 'pre_execution_tests_passed': True,
 'rb_fold3_statuses': {'rb_carry_share': 'PASSES_FOLD_3_POINT_GATES',
  'rb_opportunity_share': 'FAILS_FOLD_3_POINT_GATES'},
 'retired_families': ['wr_target_share', 'te_target_share']}

## Data audit

In [2]:
audit = pd.read_csv(OUT / 'data_audit_2023.csv')
joins = pd.read_csv(OUT / 'join_coverage_2023.csv')
assert audit.at[0, 'duplicate_key_rows'] == 0
assert audit.at[0, 'required_null_cells'] == 0
assert (joins['matched_rows'] == joins['rows']).all()
display(audit, joins)

,season,canonical_rows,unique_players,played_games,observed_weeks,duplicate_key_rows,duplicate_key_rate,required_null_cells,required_null_rows,identity_resolved_rows,identity_coverage,quality_pass_rows,quality_pass_rate,qualifying_rows,qualifying_rate
0,2023,7448,531,272,18,0,0.0,0,0,7448,1.0,7448,1.0,7448,1.0


,season,join,rows,matched_rows,coverage_rate
0,2023,opportunity_to_identity,31567,31567,1.0
1,2023,participating_player_to_identity,10225,10225,1.0


## RB family results and locked statuses

In [3]:
comparisons = pd.read_csv(OUT / 'rb_family_comparisons_2023.csv')
primary = comparisons.query("partial_policy == 'PRIMARY_CONFIRMED_EXCLUDED'")
gates = pd.read_csv(OUT / 'fold3_gate_decisions.csv')
display(primary, gates)

,partial_policy,role_family,full_alerts,naive_alerts,full_evaluable_alerts,naive_evaluable_alerts,full_precision,naive_precision,precision_improvement,relative_precision_improvement,precision_improvement_ci_low,precision_improvement_ci_high,full_reversion_rate,naive_reversion_rate,reversion_improvement,full_median_retention,naive_median_retention
2,PRIMARY_CONFIRMED_EXCLUDED,rb_carry_share,60,60,47,49,0.659574,0.530612,0.128962,0.243044,-0.030642,0.263314,0.230769,0.326923,0.096154,0.794931,0.526657
3,PRIMARY_CONFIRMED_EXCLUDED,rb_opportunity_share,74,74,57,58,0.771930,0.568966,0.202964,0.356725,0.057772,0.344047,0.156250,0.317460,0.161210,0.910085,0.616665


,role_family,candidate_disposition,fold3_candidate_status,descriptive_locked_point_status,alerts,evaluable_alerts,precision,naive_precision,precision_improvement,reversion_rate,...,check_min_holdout_alerts,check_min_persistence_precision,check_min_absolute_improvement_vs_naive,check_max_immediate_reversion_rate,check_min_reversion_improvement_vs_naive,check_min_median_retention,check_min_alerts_per_week,check_direction_consistent_across_periods,check_frozen_before_holdout,failed_checks
0,rb_carry_share,PRIMARY_CANDIDATE,PASSES_FOLD_3_POINT_GATES,PASSES_FOLD_3_POINT_GATES,60,47,0.659574,0.530612,0.128962,0.230769,...,True,True,True,True,True,True,True,True,True,NaN
1,rb_opportunity_share,SHADOW_CANDIDATE,FAILS_FOLD_3_POINT_GATES,FAILS_FOLD_3_POINT_GATES,74,57,0.771930,0.568966,0.202964,0.156250,...,True,True,True,True,True,True,True,False,True,direction_consistent_across_periods
2,wr_target_share,RETIRED_DESCRIPTIVE_ONLY,NOT_APPLICABLE_RETIRED,FAILS_FOLD_3_POINT_GATES,27,23,0.260870,0.217391,0.043478,0.423077,...,False,False,False,False,True,False,True,False,True,min_holdout_alerts | min_persistence_precision...
3,te_target_share,RETIRED_DESCRIPTIVE_ONLY,NOT_APPLICABLE_RETIRED,INSUFFICIENT_EVIDENCE,4,4,0.500000,1.000000,-0.500000,0.250000,...,False,False,False,True,False,False,False,False,True,min_holdout_alerts | min_persistence_precision...


## Direction and weekly stability

In [4]:
direction = pd.read_csv(OUT / 'cross_season_direction_2021_2023.csv')
weekly = pd.read_csv(OUT / 'cross_season_weekly_2021_2023.csv')
display(direction[direction.role_family.str.startswith('rb_')], weekly[weekly.role_family.str.startswith('rb_')])

,period,role_family,direction,alerts_full,evaluable_alerts_full,precision_full,reversion_rate_full,median_retention_full,alerts_naive,evaluable_alerts_naive,precision_naive,reversion_rate_naive,median_retention_naive,precision_improvement,reversion_improvement
0,redeveloped_2021,rb_carry_share,decrease,24,19,0.789474,0.157895,0.864068,27.0,17.0,0.588235,0.200000,0.773807,0.201238,0.042105
1,redeveloped_2021,rb_carry_share,increase,32,24,0.708333,0.206897,0.738174,29.0,24.0,0.416667,0.370370,0.318850,0.291667,0.163474
2,redeveloped_2021,rb_opportunity_share,decrease,33,22,0.636364,0.115385,0.633222,34.0,23.0,0.652174,0.148148,0.802114,-0.015810,0.032764
3,redeveloped_2021,rb_opportunity_share,increase,44,33,0.696970,0.243243,0.715135,43.0,32.0,0.437500,0.416667,0.385041,0.259470,0.173423
8,untouched_2022,rb_carry_share,decrease,27,21,0.476190,0.238095,0.495693,24.0,18.0,0.444444,0.421053,0.277651,0.031746,0.182957
9,untouched_2022,rb_carry_share,increase,22,18,0.833333,0.052632,0.791246,25.0,20.0,0.500000,0.380952,0.462125,0.333333,0.328321
10,untouched_2022,rb_opportunity_share,decrease,32,24,0.541667,0.185185,0.595091,32.0,23.0,0.434783,0.400000,0.273732,0.106884,0.214815
11,untouched_2022,rb_opportunity_share,increase,27,23,0.695652,0.086957,0.844571,27.0,22.0,0.636364,0.227273,0.612591,0.059289,0.140316
16,untouched_2023,rb_carry_share,decrease,31,25,0.600000,0.259259,0.702476,27.0,22.0,0.500000,0.347826,0.427087,0.100000,0.088567
17,untouched_2023,rb_carry_share,increase,29,22,0.727273,0.200000,0.829902,33.0,27.0,0.555556,0.310345,0.526657,0.171717,0.110345


,period,role_family,seasons,season_weeks,weekly_median,weekly_maximum,zero_alert_weeks,active_weeks,weekly_mean
0,redeveloped_2021,rb_carry_share,2021,18,3.0,8,5,13,3.111111
1,redeveloped_2021,rb_opportunity_share,2021,18,4.0,11,5,13,4.277778
4,untouched_2022,rb_carry_share,2022,18,2.5,7,5,13,2.722222
5,untouched_2022,rb_opportunity_share,2022,18,3.5,9,6,12,3.277778
8,untouched_2023,rb_carry_share,2023,18,4.0,8,5,13,3.333333
9,untouched_2023,rb_opportunity_share,2023,18,4.5,11,5,13,4.111111


## Partial-game sensitivity and pooled untouched evidence

In [5]:
sensitivity = pd.read_csv(OUT / 'partial_game_sensitivity_2023.csv')
pooled = pd.read_csv(OUT / 'pooled_untouched_family_2022_2023.csv')
display(sensitivity[sensitivity.role_family.str.startswith('rb_')], pooled[pooled.role_family.str.startswith('rb_')])

,partial_policy,role_family,full_alerts,naive_alerts,full_evaluable_alerts,naive_evaluable_alerts,full_precision,naive_precision,precision_improvement,relative_precision_improvement,...,full_median_retention,naive_median_retention,sensitivity_type,delta_vs_primary_full_alerts,delta_vs_primary_full_evaluable_alerts,delta_vs_primary_full_precision,delta_vs_primary_precision_improvement,delta_vs_primary_full_reversion_rate,delta_vs_primary_reversion_improvement,delta_vs_primary_full_median_retention
0,ALL_INCLUDED,rb_carry_share,61,61,47,49,0.659574,0.551020,0.108554,0.197006,...,0.754819,0.527938,confirmed_partial_inclusion_sensitivity,1,0,0.000000,-0.020408,0.014514,-0.020682,-0.040112
1,ALL_INCLUDED,rb_opportunity_share,75,75,58,58,0.775862,0.568966,0.206897,0.363636,...,0.896747,0.616665,confirmed_partial_inclusion_sensitivity,1,1,0.003932,0.003932,-0.002404,-0.018181,-0.013339
4,PRIMARY_CONFIRMED_EXCLUDED,rb_carry_share,60,60,47,49,0.659574,0.530612,0.128962,0.243044,...,0.794931,0.526657,primary,0,0,0.000000,0.000000,0.000000,0.000000,0.000000
5,PRIMARY_CONFIRMED_EXCLUDED,rb_opportunity_share,74,74,57,58,0.771930,0.568966,0.202964,0.356725,...,0.910085,0.616665,primary,0,0,0.000000,0.000000,0.000000,0.000000,0.000000
8,STRICT_SUSPECTED_EXCLUDED,rb_carry_share,59,59,48,49,0.645833,0.530612,0.115221,0.217147,...,0.791804,0.526657,suspected_partial_exclusion_sensitivity,-1,1,-0.013741,-0.013741,0.000000,0.000000,-0.003127
9,STRICT_SUSPECTED_EXCLUDED,rb_opportunity_share,71,71,55,57,0.763636,0.578947,0.184689,0.319008,...,0.930276,0.635986,suspected_partial_exclusion_sensitivity,-3,-2,-0.008293,-0.018275,-0.013393,0.002384,0.020191


,period,role_family,full_alerts,naive_alerts,full_evaluable_alerts,naive_evaluable_alerts,full_precision,naive_precision,precision_improvement,relative_precision_improvement,precision_improvement_ci_low,precision_improvement_ci_high,full_reversion_rate,naive_reversion_rate,reversion_improvement,full_median_retention,naive_median_retention,precision_ci_low,precision_ci_high
0,pooled_untouched_2022_2023,rb_carry_share,109,109,86,87,0.651163,0.505747,0.145416,0.287526,0.035280,0.249333,0.195652,0.358696,0.163043,0.738639,0.504611,0.546512,0.744186
1,pooled_untouched_2022_2023,rb_opportunity_share,133,133,104,103,0.701923,0.553398,0.148525,0.268387,0.023946,0.261304,0.149123,0.318182,0.169059,0.793768,0.587650,0.615385,0.788462


## Equal-volume and temporal validation

In [6]:
equal = pd.read_csv(OUT / 'equal_volume_verification_2023.csv')
temporal = pd.read_csv(OUT / 'temporal_integrity_checks_2023.csv')
assert bool(equal['equal_volume'].all())
assert bool(temporal['passed'].all())
print(f'Equal-volume cells: {len(equal)}; temporal checks: {len(temporal)}; all passed.')
display(temporal)

Equal-volume cells: 216; temporal checks: 8; all passed.


,check,passed
0,only_2023_alerts,True
1,minimum_four_game_baseline,True
2,confirmation_window_complete,True
3,baseline_strictly_before_confirmation,True
4,confirmation_ends_on_alert_week,True
5,first_outcome_strictly_after_alert,True
6,second_outcome_strictly_after_first,True
7,same_season_grouping_enabled_by_frozen_config,True


## Interpretation

RB carry passes the unchanged Fold 3 point gates and is recommended to advance unchanged. RB opportunity remains shadow because the locked cross-period direction gate fails. Neither result is a validation claim; WR/TE remain retired.